## Methodology: Three-Sample Procedure

Three different sets of samples to avoid potential issues of **bias and overfitting**
that are already well known in the Longstaff-Schwartz pricing method alone.

### Sample splitting

| Sample | Size | Purpose |
|--------|------|---------|
| First | $Q_1$ | Compute coefficients $\alpha_n^{Q_1}$ for $n \in \{1,\dots,N\}$, defining the approximation $\hat{M}_n$ |
| Second | $Q_2$ | Run Longstaff-Schwartz to obtain $\hat{\tau}^{LS_i, Q_2}$, $i \in \{1,2\}$ |
| Third | $Q_3$ | Out-of-sample Monte Carlo estimation of $\mathbb{E}[Z_{\hat{\tau}^{LS_i, Q_2}}]$ |

### Control variate estimator

On the third sample, we apply control variate variance reduction and compute:

$$\frac{1}{Q_3} \sum_{q=1}^{Q_3} Z_{\hat{\tau}^{LS_i, Q_2,(q)}}^{(q)} - \hat{\lambda} \hat{M}_{\hat{\tau}^{LS_i, Q_2,(q)}}^{(q)}$$

where

$$\hat{\lambda} = \frac{\sum_{q=1}^{Q_3} Z_{\hat{\tau}^{LS_i,Q_2,(q)}}^{(q)} \, \hat{M}_{\hat{\tau}^{LS_i,Q_2,(q)}}^{(q)}}{\sum_{q=1}^{Q_3} \left(\hat{M}_{\hat{\tau}^{LS_i,Q_2,(q)}}^{(q)}\right)^2}$$

minimizes the asymptotic variance. Here $Q_3$ denotes the number of samples in the second and
third sets used for the Longstaff-Schwartz algorithm and the Monte Carlo estimation respectively.

In [1]:
from code_utils import *
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def ls_train_alfonsi(K, N, N_bar, P, asset, discounted_asset, basis, discount_factors, coefficients, paths, exercise_indices, max_deg):
    # Determining the discounted paygoff
    asset_exercise = asset[list(exercise_indices),:] # Shape (N, npaths_training)
    intrinsic_value = np.maximum(0, K - asset_exercise) * discount_factors[:,np.newaxis] 
    # Determining the martingale using the Alfonsi scheme for the modified continuation value
    martingale = alfonsi_martingale(N, N_bar, P, discounted_asset, coefficients, paths, asset ) # Shape (N, paths)

    # Training phase : Backward induction compute the coefficients
    value = intrinsic_value[-1,:] - martingale[-1, :] # Modified payoff 
    
    train_deg = np.zeros(N-1, dtype=int) # Actual degree used for regression at each time step
    reg_coefficients = np.zeros((N - 1, max_deg + 1)) 
    itm = intrinsic_value > 0 # We are not modified the ITM paths since we just modified the continuation value bc of (7)

    for t in range(N - 1, 0, -1):
        itm_paths = np.sum(itm[t]) # Number of in the money paths at time t
        train_deg[t-1] = min(max_deg, int(itm_paths / 100) ) 
        # " Better to use (7) instead of (4)"
        target = value + martingale[t-1,:]

        if (train_deg[t-1] > 0) & (itm_paths > train_deg[t-1]):
            Y_itm = target[itm[t]]
            vander_full = basis[t-1, :, :train_deg[t-1]+1]
            X_itm_vander = basis[t-1, itm[t] , :train_deg[t-1]+1]
            reg_coefficients[t-1, :train_deg[t-1]+1], _, _, _ = np.linalg.lstsq(X_itm_vander, Y_itm, rcond=None) 
            continuation = vander_full @ reg_coefficients[t-1, :train_deg[t-1]+1]
            
            exercise = (intrinsic_value[t,:]  > continuation) & itm[t]
            
            value = np.where(exercise, intrinsic_value[t,:] - martingale[t-1, :], value)

    return reg_coefficients, train_deg


def ls_apply_alfonsi(K, N, N_bar, P, asset, discounted_asset, basis, discount_factors, coefficients_alfonsi, regression_coefs, paths, exercise_indices, max_deg, train_deg):

    asset_exercise = asset[list(exercise_indices), :]
    intrinsic_value = (np.maximum(0, K - asset_exercise)* discount_factors[:, np.newaxis])

    martingale = alfonsi_martingale( N, N_bar, P, discounted_asset, coefficients_alfonsi, paths, asset)
    matrix_exercise_time = np.zeros((N, paths), dtype=bool)
    has_exercised = np.zeros(paths, dtype=bool)

    itm = intrinsic_value > 0
    for t in range(0, N-1):
        itm_paths = np.sum(itm[t+1]) # Number of in the money paths at time t
        if (train_deg[t] > 0) & (itm_paths > train_deg[t]):
            vander = basis[t,:,:train_deg[t]+1]
            continuation = vander @ regression_coefs[t,:train_deg[t]+1]
            exercise_condition = (intrinsic_value[t+1, :] > continuation) & (~has_exercised) & itm[t+1]
            matrix_exercise_time[t, exercise_condition] = True
            has_exercised |= exercise_condition

    matrix_exercise_time[-1, ~has_exercised] = True

    return matrix_exercise_time, martingale

In [3]:
# Alfonsi + LS

# Paramters
N = 50    # Number of exercising dates
N_bar = 5     # Number of intervals between two exercising dates
K = 100   # Strike price
r = 0.05  # Risk-free rate
sigma = 0.3   # Volatility
T = 0.2   # Maturity
seed = 42    # Seed for reproducibility
P = 50

max_deg = 5 # Maximum degree for the polynomial basis in regression
paths_alfonsi = 10**5
paths_ls_train = 10**5
paths_ls_pricing = 10**5

real_prices = np.array([
    40.00000000, 39.00000000, 38.00000000, 37.00000000, 36.00000000,
    35.00000000, 34.00000000, 33.00000000, 32.00000000, 31.00000000,
    30.00000000, 29.00000000, 28.00000000, 27.00000000, 26.00000000,
    25.00000000, 24.00000000, 23.00000000, 22.00000000, 21.00000000,
    20.00070685, 19.01347004, 18.04313091, 17.09109638, 16.15883366,
    15.24783858, 14.35960625, 13.49560169, 12.65723381, 11.84582143,
    11.06257234, 10.30856320,  9.58471980,  8.89180771,  8.23041673,
     7.60094690,  7.00361025,  6.43844700,  5.90532237,  5.40390521,
     4.93370425,  4.49410547,  4.08432251,  3.70344748,  3.35050142,
     3.02435964,  2.72386986,  2.44779995,  2.19486933,  1.96380311,
     1.75326609,  1.56197956,  1.38862261,  1.23195119,  1.09070376,
     0.96370545,  0.84978944,  0.74786820,  0.65689777,  0.57588855,
     0.50392803,  0.44013960,  0.38373063,  0.33395543,  0.29012326,
     0.25161274,  0.21784490,  0.18829301,  0.16248633,  0.13999277,
     0.12042199,  0.10342780,  0.08869869,  0.07595400,  0.06494518,
     0.05545341,  0.04728321,  0.04026147,  0.03423616,  0.02907429,
     0.02465918
])

prices = np.empty(81)
uncertainty = np.empty(81)
percentage_error = np.empty(81)

for S0 in range(60,141):

    # Temporal grids
    exercise_time = np.linspace(0, T, N + 1)
    exercise_indices = range(0, N_bar * N + 1, N_bar)
    dense_grid = np.linspace(0, T, N_bar * N + 1)

    discount_factor = np.exp(-r * dense_grid)
    discount_factor_exercise_time = np.exp(-r * exercise_time)

    # Step 1 : determining the martingale for the control variate using Alfonsi's scheme
    asset_alfonsi = S0 * simulate_gbm_nv(r, sigma, T, dense_grid[1] - dense_grid[0], paths_alfonsi, seed)
    discounted_asset_alfonsi = asset_alfonsi * discount_factor[:, np.newaxis]
    discounted_payoff_alfonsi= (np.maximum(K - asset_alfonsi[exercise_indices, :], 0) * discount_factor_exercise_time[:, np.newaxis])
    #Determining the martingale
    coefficients_alfonsi, _ = computing_coefficients_martingale_alfonsi(N, N_bar, P, discounted_asset_alfonsi, discounted_payoff_alfonsi, paths_alfonsi, asset_alfonsi)


    # Step 2 : Longstaff-Schwartz method to get the optimal stopping rule
    asset_ls = S0 * simulate_gbm_nv(r, sigma, T, dense_grid[1] - dense_grid[0], paths_ls_train, seed + 1)
    discounted_asset_ls = asset_ls * discount_factor[:, np.newaxis]

    ls_log = transformation(asset_ls)
    ls_log_ex = ls_log[list(exercise_indices), :]      # (N+1, paths): righe = T_0..T_N
    basis_ls = np.ones((N-1, paths_ls_train, max_deg + 1))
    for d in range(1, max_deg + 1):
        basis_ls[:, :, d] = ls_log_ex[1:N, :] * basis_ls[:, :, d-1]   # righe T_1..T_{N-1}
        
    regression_coefs, train_deg = ls_train_alfonsi(K, N, N_bar, P, asset_ls, discounted_asset_ls, basis_ls, discount_factor_exercise_time, coefficients_alfonsi, paths_ls_train, exercise_indices, max_deg)

    # Step 3 : Pricing the option using the optimal stopping rule and the martingale as control variate
    asset_pricing = S0 * simulate_gbm_nv(r, sigma, T, dense_grid[1] - dense_grid[0], paths_ls_pricing, seed + 2)
    discounted_asset_pricing = asset_pricing * discount_factor[:, np.newaxis]
    discounted_payoff_pricing = (np.maximum(K - asset_pricing[exercise_indices, :], 0) * discount_factor_exercise_time[:, np.newaxis])

    ls_price   = transformation(asset_pricing)
    ls_price_ex = ls_price[list(exercise_indices), :]
    basis_pricing = np.ones((N-1, paths_ls_pricing, max_deg + 1))
    for d in range(1, max_deg + 1):
        basis_pricing[:, :, d] = ls_price_ex[1:N, :] * basis_pricing[:, :, d-1]
        
    optimal_stopping, martingale_pricing = ls_apply_alfonsi( K, N, N_bar, P, asset_pricing, discounted_asset_pricing, basis_pricing, discount_factor_exercise_time, coefficients_alfonsi, regression_coefs, paths_ls_pricing, exercise_indices, max_deg, train_deg)

    # Determining the coefficient for the control variate
    payoff_control_variate = np.sum(discounted_payoff_pricing[1:] * optimal_stopping,axis=0) # Payoff of the control variate using the optimal stopping rule
    martingale_control_variate = np.sum(martingale_pricing * optimal_stopping,axis=0) # Martingale control variate using the optimal stopping rule
    coeff_lambda = np.sum(payoff_control_variate * martingale_control_variate) / np.sum(martingale_control_variate**2) # Coefficient for the control variate

    #modified_payoff = payoff_control_variate - coeff_lambda * martingale_control_variate
    #antithetic_sum = (modified_payoff[:paths_ls_pricing//2] + modified_payoff[paths_ls_pricing//2:]) / 2
    #price_mc = np.mean(antithetic_sum)
    #std_mc = np.std(antithetic_sum, ddof=1) / np.sqrt(paths_ls_pricing//2)
    price_mc  = np.mean(payoff_control_variate - coeff_lambda * martingale_control_variate)
    std_mc = np.std(payoff_control_variate - coeff_lambda * martingale_control_variate, ddof=1) / np.sqrt(paths_ls_pricing)
    #std_mc = np.std(payoff_control_variate - coeff_lambda * martingale_control_variate) / np.sqrt(paths_ls_pricing)
    #std_mc = np.std(payoff_control_variate - coeff_lambda * martingale_control_variate, ddof=1) / np.sqrt(paths_ls_pricing)

    #uncertainty = np.std(antithetic_sum, ddof=1) / np.sqrt(npaths_pricing//2)


    # # Since we are generating the paths using a variance reduction technique (antithetic variates), we need to take
    # # this into account: we need to do the average of the "antithetic values", hence this is wrong option_price = np.mean(exercise_payoff)
    # antithetic_sum = (exercise_payoff[:npaths_pricing//2] + exercise_payoff[npaths_pricing//2:]) / 2
    # option_price = np.mean(antithetic_sum)

    # immediate_exercise = max(0, K - S0) # Immediate exercise value at time 0
    # option_price = max(option_price, immediate_exercise)

    # uncertainty = np.std(antithetic_sum, ddof=1) / np.sqrt(npaths_pricing//2)

    #
    immediate_exercise = max(K - S0, 0)
    standard_error = std_mc
    price = price_mc

    if immediate_exercise >= price_mc:
        price = immediate_exercise
        standard_error = 0.0       
    else:
        price = price_mc
        standard_error = std_mc
    
    # Results
    prices[S0-60] = price
    uncertainty[S0-60] = standard_error
    percentage_error[S0-60] = 100 * abs(price - real_prices[S0-60]) / real_prices[S0-60]
    print(f"S0= {S0}, Estimated Price: {prices[S0-60]:.5f}, Real Price: {real_prices[S0-60]:.5f}, Percentage Error: {percentage_error[S0-60]:.2f}%, Uncertainty: {uncertainty[S0-60]:.5f}")
    

S0= 60, Estimated Price: 40.00000, Real Price: 40.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 61, Estimated Price: 39.00000, Real Price: 39.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 62, Estimated Price: 38.00000, Real Price: 38.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 63, Estimated Price: 37.00000, Real Price: 37.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 64, Estimated Price: 36.00000, Real Price: 36.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 65, Estimated Price: 35.00000, Real Price: 35.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 66, Estimated Price: 34.00000, Real Price: 34.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 67, Estimated Price: 33.00000, Real Price: 33.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 68, Estimated Price: 32.00000, Real Price: 32.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 69, Estimated Price: 31.00000, Real Price: 31.00000, Percentage Error

In [4]:
np.savez('lower_bound_ls_alfonsi_results.npz', S0=np.arange(60,141), price=prices, error=percentage_error, uncertainty=uncertainty)